In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "02. prot_clustering"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import seaborn as sns

# Add the 'src' directory to the system path
src_dir = os.path.abspath(os.path.join(os.getcwd(), "src"))
sys.path.insert(0, src_dir)

#import the module
import corr_utils

import warnings
warnings.filterwarnings("ignore")
from joblib import dump, load
from scipy.cluster.hierarchy import fcluster
import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import dendrogram
from sklearn.utils import resample
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.spatial.distance import pdist, squareform
import community as community_louvain

from sklearn.decomposition import PCA

import networkx as nx

## Protein - Protein Correlation Clustering

In [ ]:
proteins = load('data/full_prot.joblib')
proteins

Clustermap

In [ ]:
#row_linkage1 = g.dendrogram_row.linkage
row_linkage1 = sch.linkage(proteins, method="ward", metric="euclidean")
corr_utils.plot_dendrogram(row_linkage1)

In [ ]:
os.makedirs('results/clustermaps', exist_ok=True)

# Create the clustermap.
sns.set_theme(style="white")
g = sns.clustermap(
    proteins,
    method="ward",         # Clustering method
    metric="euclidean",    # Distance metric
    cmap="coolwarm",           # Color map for correlation data
    center=0,              # Center the colormap at zero
    vmin=-1, vmax=1,       # Expected correlation range
    standard_scale=1,      # Standardize each row (protein)
    figsize=(10, 10)       # Figure size
)

# Rotate x-axis labels for readability.
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")
g.savefig('results/clustermaps/map_pp_raw.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Takes ~ 80 min to run, so commented out and keep the first one as reference
# # Create the clustermap.
# sns.set_theme(style="white")
# g = sns.clustermap(
#     proteins,
#     method="ward",         # Clustering method
#     metric="euclidean",    # Distance metric
#     cmap="coolwarm",           # Color map for correlation data
#     center=0,              # Center the colormap at zero
#     vmin=-1, vmax=1,       # Expected correlation range
#     #standard_scale=1,      # Standardize each row (protein)
#     figsize=(10, 10)       # Figure size
# )

# # 5. Rotate x-axis labels for readability.
# plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")
# g.savefig('results/clustermaps/map_pp_std.png', dpi=300, bbox_inches='tight')

# plt.show()

#### Save clusters

In [ ]:
# 1. Elbow Method
elbow_k = corr_utils.elbow_method(proteins, row_linkage1)

# 2. Silhouette Score Method
silhouette_k = corr_utils.silhouette_method(proteins, row_linkage1) 

# Uncomment to save cluster assignments for all k from 2 to silhouette_k
# Create the directory if it doesn't exist
os.makedirs("results/clusters_pp", exist_ok=True)

# Save cluster assignments for all values from 2 to 4
for k in range(2, 5):
    # Get cluster labels for this k
    cluster_labels = fcluster(row_linkage1, k, criterion='maxclust')
    
    # Create a DataFrame mapping each protein to its cluster
    clusters_df = pd.DataFrame({
        'Protein': proteins.index,
        'Cluster': cluster_labels
    })
    
    # Define the output file path
    output_file = f"results/clusters_pp/prot_prot_clusters_k{k}.csv"
    
    # Save the cluster assignments to a CSV file
    clusters_df.to_csv(output_file, index=False)
    
    print(f"Saved cluster assignments for k={k} to '{output_file}'")

print(f"\nAll cluster assignments from k=2 to k={silhouette_k} have been saved to the 'results/clusters_pp' folder.")

In [ ]:
clusters = sch.fcluster(row_linkage1, 2, criterion="maxclust")

PCA

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()  # flatten to easily iterate over

# Loop over cluster numbers from 2 to 10
for i, k in enumerate(range(2, 8)):
    # Compute clusters using hierarchical clustering
    clusters = sch.fcluster(row_linkage1, k, criterion="maxclust")
    
    # Perform PCA on the cleaned data
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(proteins.values)
    
    # Create DataFrame for PCA results with cluster assignments
    pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"], index=proteins.index)
    pca_df["Cluster"] = clusters
    
    # Plot on the corresponding axis
    ax = axes[i]
    sns.scatterplot(
        data=pca_df, 
        x="PC1", 
        y="PC2", 
        hue="Cluster", 
        palette="Set1", 
        s=10, 
        ax=ax,
        legend=False  # Disable legend for individual subplots
    )
    ax.set_title(f"k = {k}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()

## Protein Expression Clustering

In [ ]:
# Original proteomics dataset
protein_expr = load('data/proteomics_matched.joblib')
protein_expr = protein_expr.T

In [ ]:
protein_expr

In [ ]:
row_linkage2 = sch.linkage(protein_expr, method="ward", metric="euclidean")

corr_utils.plot_dendrogram(row_linkage2)

In [ ]:

# Create directory if it doesn't exist
os.makedirs('results/clustermaps', exist_ok=True)

# Create the clustermap
sns.set_theme(style="white")
g = sns.clustermap(
    protein_expr,
    method="ward",         # Clustering method
    metric="euclidean",    # Distance metric
    cmap="coolwarm",       # Color map for correlation data
    center=0,              # Center the colormap at zero
    vmin=-1, vmax=1,       # Expected correlation range
    standard_scale=1,      # Standardize each row (protein)
    figsize=(10, 10)       # Figure size
)

# Rotate x-axis labels for readability
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")

# Save the figure
g.savefig('results/clustermaps/map_pe_raw.png', dpi=300, bbox_inches='tight')

# Show the plot (optional - you can comment this out if running in a script)
plt.show()

In [ ]:
# Takes ~ 80 min to run, so commented out and keep the first one as reference

# # Create the clustermap
# sns.set_theme(style="white")
# g = sns.clustermap(
#     protein_expr,
#     row_linkage=row_linkage2,
#     method="ward",         # Clustering method
#     metric="euclidean",    # Distance metric
#     cmap="coolwarm",       # Color map for correlation data
#     center=0,              # Center the colormap at zero
#     vmin=-1, vmax=1,       # Expected correlation range
#     #standard_scale=1,      # Standardize each row (protein)
#     figsize=(10, 10)       # Figure size
# )

# # Rotate x-axis labels for readability
# plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")

# # Save the figure
# g.savefig('results/clustermaps/map_pe_std.png', dpi=300, bbox_inches='tight')

# # Show the plot (optional - you can comment this out if running in a script)
# plt.show()

In [ ]:
row_linkage2 = g.dendrogram_row.linkage

#### Save clusters

In [ ]:
# Elbow Method
elbow_k = corr_utils.elbow_method(protein_expr, row_linkage2)

# Silhouette Score Method
silhouette_k = corr_utils.silhouette_method(protein_expr, row_linkage2)

# Uncomment to save cluster assignments for all k from 2 to silhouette_k
# Extracting Cluster Assignments for multiple k values

# Create the directory if it doesn't exist
os.makedirs("results/clusters_pe", exist_ok=True)

# Save cluster assignments for all values from 2 to 4
for k in range(2, 5):
    # Get cluster labels for this k
    cluster_labels = fcluster(row_linkage2, k, criterion='maxclust')
    
    # Create a DataFrame mapping each protein to its cluster
    clusters_df = pd.DataFrame({
        'Protein': protein_expr.index,
        'Cluster': cluster_labels
    })
    
    # Define the output file path
    output_file = f"results/clusters_pe/prot_expr_clusters_k{k}.csv"
    
    # Save the cluster assignments to a CSV file
    clusters_df.to_csv(output_file, index=False)
    
    print(f"Saved cluster assignments for k={k} to '{output_file}'")

print(f"\nAll cluster assignments from k=2 to k={silhouette_k} have been saved to the 'results_pe' folder.")

PCA

In [ ]:

clusters = sch.fcluster(row_linkage2, 2, criterion="maxclust")
# Create a 2x5 grid for plots with a reduced overall figure size
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()  # flatten to easily iterate over
# Loop over cluster numbers from 2 to 10
for i, k in enumerate(range(2, 8)):
    # Compute clusters using hierarchical clustering
    clusters = sch.fcluster(row_linkage2, k, criterion="maxclust")
    
    # Perform PCA on the cleaned data
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(protein_expr.values)
    
    # Create DataFrame for PCA results with cluster assignments
    pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"], index=protein_expr.index)
    pca_df["Cluster"] = clusters
    
    # Plot on the corresponding axis
    ax = axes[i]
    sns.scatterplot(
        data=pca_df, 
        x="PC1", 
        y="PC2", 
        hue="Cluster", 
        palette="Set1", 
        s=10, 
        ax=ax,
        legend=False  # Disable legend for individual subplots
    )
    ax.set_title(f"k = {k}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()

In [ ]:
clusters = sch.fcluster(row_linkage2, 3, criterion="maxclust")

In [ ]:
# Correlation Network Analysis

# Build a network where nodes are proteins and an edge exists if the similarity between their gene correlation profiles is high.
# Here, compute the protein–protein correlation matrix using the rows of pearson_clean.
protein_corr = protein_expr.T.corr()  # proteins x proteins

# Set a correlation threshold for connecting nodes 
corr_threshold = 0.7

# Create an undirected graph
G = nx.Graph()

# Map each protein (row) to its cluster label
protein_cluster_mapping = dict(zip(protein_expr.index, clusters))

# Add nodes with cluster attribute
for protein in protein_corr.index:
    G.add_node(protein, cluster=protein_cluster_mapping[protein])

# Add edges between proteins with correlation above the threshold
for i, protein_i in enumerate(protein_corr.index):
    for j, protein_j in enumerate(protein_corr.columns):
        if i < j and protein_corr.loc[protein_i, protein_j] > corr_threshold:
            G.add_edge(protein_i, protein_j, weight=protein_corr.loc[protein_i, protein_j])

# Compute network density
density = nx.density(G)
print("\nNetwork Density:", density)

# For modularity, we can use the community_louvain package (install via pip install python-louvain) 
partition = community_louvain.best_partition(G)
modularity = community_louvain.modularity(partition, G)
print("Network Modularity:", modularity)

# Identify hub proteins (top 5 by degree centrality)
degree_centrality = nx.degree_centrality(G)
sorted_hubs = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)
print("\nTop 5 Hub Proteins (by Degree Centrality):")
for protein, centrality in sorted_hubs[:5]:
    print(f"{protein}: {centrality:.4f}")

# Optionally, visualize the network (this can be messy for large networks)
plt.figure(figsize=(10,8))
pos = nx.spring_layout(G, seed=42)  # positions for all nodes
node_colors = [G.nodes[node]['cluster'] for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_size=50, cmap=plt.cm.Set1, node_color=node_colors)
nx.draw_networkx_edges(G, pos, alpha=0.3)
plt.title("Protein Correlation Network (Edges with r > {:.2f})".format(corr_threshold))
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Create zoomed version of central cluster
plt.figure(figsize=(12, 10))

# Get the same layout as the original visualization for consistency
pos = nx.spring_layout(G, seed=42)  # Use same seed as original for consistency

# Get the approximate center coordinates
x_coords = [coord[0] for coord in pos.values()]
y_coords = [coord[1] for coord in pos.values()]
center_x = sum(x_coords) / len(x_coords)
center_y = sum(y_coords) / len(y_coords)

# Define radius for central region
radius = 0.3  # Adjust this value to control zoom level

# Identify nodes in the central region
central_nodes = [node for node in G.nodes() if 
                ((pos[node][0] - center_x)**2 + (pos[node][1] - center_y)**2) < radius**2]

# Draw edges with increased transparency for better visibility
central_edge_list = [(u, v) for u, v in G.edges() if u in central_nodes and v in central_nodes]
central_G = nx.Graph()
central_G.add_edges_from(central_edge_list)
nx.draw_networkx_edges(central_G, pos, alpha=0.2, width=0.5)

# Draw high correlation edges more prominently
high_corr_edges = [(u, v) for u, v, d in G.edges(data=True) 
                  if d['weight'] > 0.8 and u in central_nodes and v in central_nodes]
high_corr_G = nx.Graph()
high_corr_G.add_edges_from(high_corr_edges)
nx.draw_networkx_edges(high_corr_G, pos, alpha=0.6, width=1.0, edge_color='blue')

# Draw nodes for central region
central_nodes_colors = [G.nodes[node]['cluster'] for node in central_nodes]
nx.draw_networkx_nodes(G, pos, nodelist=central_nodes, node_size=15, 
                      alpha=0.8, cmap=plt.cm.Set1, node_color=central_nodes_colors)

# Add labels for hub proteins in central region (top 10)
top_hubs = [protein for protein, _ in sorted_hubs[:10]]
#central_hub_labels = {node: node for node in top_hubs if node in central_nodes}
#nx.draw_networkx_labels(G, pos, labels=central_hub_labels, font_size=10, font_color='black', font_weight='bold')

# Set plot limits for zoom effect
plt.xlim(center_x - radius, center_x + radius)
plt.ylim(center_y - radius, center_y + radius)

plt.title("Zoomed View of Central Cluster (High-Correlation Proteins)")
plt.axis("off")
plt.tight_layout()
plt.savefig("results/clusters_pe/zoomed_central_cluster.png", dpi=300)
plt.show()